# 08.05 · Subgraphs

**Module 08 · LangGraph Production Features** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 90–120 min |
| 📋 Prerequisites | Lessons 08.01–08.03; Module 07 (reducers, `Command`). |
| 🔑 Keys | none |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-12 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Compose** graphs: add a compiled subgraph as a node (shared state) or call it from a node (different state).
2. **Avoid** the shared-reducer trap that duplicates list items.
3. **See inside**: stream with `subgraphs=True` and read a paused subgraph's state with `get_state(subgraphs=True)`.
4. **Pause and persist** across the boundary: interrupts inside a subgraph, one checkpointer for both.

## 📚 Official reading (verified 2026-10-12)

| Page | Read it for |
|---|---|
| [Use subgraphs](https://docs.langchain.com/oss/python/langgraph/use-subgraphs) | Both ways to add a subgraph, persistence, viewing subgraph state, streaming outputs |
| [Streaming](https://docs.langchain.com/oss/python/langgraph/streaming) | `subgraphs=True` and namespaces |
| [Interrupts](https://docs.langchain.com/oss/python/langgraph/interrupts) | Interrupts inside subgraphs (the parent node re-runs too) |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain-core>=1.6.9,<2" "langgraph>=1.2.14,<2"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langgraph",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Step 1: Find the repository root (Jupyter starts in modules/08-langgraph-production-features/).
import importlib.util
import sys
from pathlib import Path

REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone.")

# Step 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY',)
# Step 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

In [ ]:
# Step 4: Silence known, harmless noise so real problems stand out.
import os
import warnings

os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")          # Chroma (Project 08 can reuse Module 06): no usage telemetry.
# Same as Lesson 02.04: langchain-core 1.6.9/1.6.10 Pydantic serializer warning about the injected runtime.
warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)

> Everything here is 🧪 offline.

---
## Part 1 · Way 1: the subgraph **is** a node (shared keys)

When parent and child share state keys, add the **compiled** subgraph with `add_node("name", subgraph)`. The child reads the shared keys from the parent's state and its updates flow back.

In [ ]:
# 🧪 Offline: a "review" subgraph shared by two pipelines.
import operator
from typing import Annotated

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt
from typing_extensions import TypedDict


class Doc(TypedDict, total=False):
    text: str
    issues: list[str]                       # no reducer: the last write wins


def spellcheck(state: Doc) -> dict:
    return {"issues": [w for w in ("teh", "recieve") if w in state["text"]]}


def fix(state: Doc) -> dict:
    return {"text": state["text"].replace("teh", "the").replace("recieve", "receive")}


review = (StateGraph(Doc).add_node("spellcheck", spellcheck).add_node("fix", fix)
          .add_edge(START, "spellcheck").add_edge("spellcheck", "fix").compile(name="review"))

parent = (StateGraph(Doc)
          .add_node("write", lambda s: {"text": "Please recieve teh new firmware."})
          .add_node("review", review)                       # the compiled subgraph as a node
          .add_edge(START, "write").add_edge("write", "review").compile())
print(parent.invoke({}))

**You should now see** `{'text': 'Please receive the new firmware.', 'issues': ['teh', 'recieve']}`: the child's `spellcheck` and `fix` updated the parent's keys.

### ⚠️ The shared-reducer trap

If a shared key has an **append** reducer, the child starts with the parent's list, appends, and returns its **whole** list as its output, which the parent's reducer appends **again**.

In [ ]:
# 🧪 Offline: `log` uses operator.add in both graphs.
class Logged(TypedDict, total=False):
    log: Annotated[list[str], operator.add]


child = StateGraph(Logged).add_node("child_step", lambda s: {"log": ["child"]}).add_edge(START, "child_step").compile()
trap = (StateGraph(Logged)
        .add_node("first", lambda s: {"log": ["parent"]})
        .add_node("child", child)
        .add_edge(START, "first").add_edge("first", "child").compile())
print(trap.invoke({"log": []})["log"])

**You should now see** `['parent', 'parent', 'child']`: the parent's item appears **twice** (measured with langgraph 1.2.14). With a longer history it gets worse at every subgraph call.

Fixes: don't share append-reduced keys with a subgraph (give the child its own key names), or use **Way 2**, below, and return only the new items. Project 08 does the latter (D-031).

---
## Part 2 · Way 2: call the subgraph from a node (different state)

When the child has **its own** schema (a private scratchpad, different key names), call it inside a node: map parent → child input, `invoke`, map child output → parent update. You control exactly what crosses the boundary.

In [ ]:
# 🧪 Offline: a research subgraph with private keys; the parent sees only `notes`.
class ResearchState(TypedDict, total=False):
    query: str
    hits: list[str]
    summary: str


def search(state: ResearchState) -> dict:
    corpus = {"battery": "Cold reduces battery life by about 40%.", "mount": "Mount at 1.5 to 2 m height."}
    return {"hits": [text for key, text in corpus.items() if key in state["query"]]}


def summarize(state: ResearchState) -> dict:
    return {"summary": " ".join(state["hits"]) or "nothing found"}


research = (StateGraph(ResearchState).add_node("search", search).add_node("summarize", summarize)
            .add_edge(START, "search").add_edge("search", "summarize").compile(name="research"))


class Main(TypedDict, total=False):
    topics: list[str]
    notes: Annotated[list[str], operator.add]


def research_all(state: Main) -> dict:
    new = [research.invoke({"query": t})["summary"] for t in state["topics"]]   # parent -> child -> parent
    return {"notes": new}                                                         # only the NEW items


main = StateGraph(Main).add_node("research_all", research_all).add_edge(START, "research_all").compile()
print(main.invoke({"topics": ["battery in winter", "mount height"], "notes": ["(earlier note)"]})["notes"])

**You should now see** `['(earlier note)', 'Cold reduces battery life by about 40%.', 'Mount at 1.5 to 2 m height.']`: no duplicates, and `hits`/`summary` never reached the parent.

| | Way 1: subgraph as a node | Way 2: call it from a node |
|---|---|---|
| State | Shares keys with the parent | Its own schema |
| Glue code | None | Map in and out |
| Good for | Reusable steps on the same state | Teams/agents with private state, calling it several times |
| Watch out for | Append reducers on shared keys | Pass `config` through if you need streaming/tracing inside |

---
## Part 3 · See inside: streaming and state with `subgraphs=True`

In [ ]:
# 🧪 Offline: updates from the parent AND the child, labelled with a namespace.
for part in parent.stream({}, stream_mode="updates", subgraphs=True, version="v2"):
    ns = part["ns"]
    where = "parent" if not ns else " > ".join(n.split(":")[0] for n in ns)
    print(f"{where:<8} {list(part['data'])[0]:<11} {part['data']}")

**You should now see** `parent write`, then `review spellcheck` and `review fix` (the child's own nodes), then `parent review` with the child's final output. `ns` is a tuple like `('review:<task id>',)`: one entry per level of nesting. Without `subgraphs=True` you'd see only the two `parent` lines.

> With **Way 2** the child's events show up too, as long as the node passes its `config` on (`research.invoke(x, config)` with a `config: RunnableConfig` parameter); LangGraph also propagates it automatically in most cases.

In [ ]:
# 🧪 Offline: an interrupt INSIDE a subgraph, one checkpointer on the parent.
def human_check(state: Doc) -> dict:
    ok = interrupt({"kind": "confirm", "text": state["text"]})
    return {} if ok == "yes" else {"text": state["text"] + " [needs work]"}


checked_review = (StateGraph(Doc).add_node("spellcheck", spellcheck).add_node("fix", fix).add_node("human_check", human_check)
                  .add_edge(START, "spellcheck").add_edge("spellcheck", "fix").add_edge("fix", "human_check")
                  .compile(name="review"))                         # no checkpointer here
pipeline = (StateGraph(Doc)
            .add_node("write", lambda s: {"text": "Please recieve teh new firmware."})
            .add_node("review", checked_review)
            .add_edge(START, "write").add_edge("write", "review")
            .compile(checkpointer=InMemorySaver()))                # the parent's checkpointer covers the child

cfg = {"configurable": {"thread_id": "sub-1"}}
paused = pipeline.invoke({}, cfg)
print("paused with:", paused["__interrupt__"][0].value)

outer = pipeline.get_state(cfg)
inner = pipeline.get_state(cfg, subgraphs=True).tasks[0].state     # the child's own snapshot
print("parent next:", outer.next, "| child next:", inner.next, "| child text:", inner.values["text"])
print(pipeline.invoke(Command(resume="no"), cfg))

**You should now see** `paused with: {'kind': 'confirm', 'text': 'Please receive the new firmware.'}`, then `parent next: ('review',) | child next: ('human_check',) | child text: Please receive the new firmware.`, and finally the text with ` [needs work]` appended.

### What just happened?

- The parent only knows "`review` is running"; `get_state(cfg, subgraphs=True)` adds the child's snapshot in `.tasks[i].state`.
- You resume **the parent** with `Command(resume=...)`; LangGraph routes the answer to the child.
- One checkpointer on the parent persists the child too (its checkpoints use the child namespace). Compile a subgraph with `checkpointer=True` only if it must **remember across calls** in the same thread (e.g. a sub-agent with its own conversation).
- On resume, the **parent node** that called the child (here: the `review` node) restarts too. With Way 2 that means code before `research.invoke(...)` re-runs: keep it idempotent (Lesson 08.03's rule, one level up).

In [ ]:
# 🧪 Offline: Project 08 uses Way 2. Its parent drawing shows one `research` node; xray shows the inside.
from mastery.graphs.research import build_research_graph, corpus_docs_search, offline_models, offline_web_search

agent = build_research_graph(offline_models(), web_search=offline_web_search(), docs_search=corpus_docs_search())
print([n for n in agent.get_graph().nodes if not n.startswith("__")])
async for part in agent.astream({"question": "How do I keep the battery healthy in winter?"},
                                {"configurable": {"thread_id": "x"}}, stream_mode="updates", subgraphs=True, version="v2"):
    if part["ns"]:
        print("  inside", part["ns"][0].split(":")[0], "->", list(part["data"]))

**You should now see** the parent nodes `['clarify_question', 'plan', 'research', 'draft', 'approval', 'finalize']`, then `inside research -> ['search_docs']` and `inside research -> ['search_web']`: the child's nodes, streamed through the wrapper node. (The run stops at `approval`, which has no checkpointer here; that's fine for a look inside.)

---
## ⚠️ Pitfalls and troubleshooting (Windows Git Bash included)

| Symptom | Cause | Fix |
|---|---|---|
| List items duplicated after a subgraph | Shared key with an append reducer (Part 1) | Different key names, or Way 2 returning only new items |
| `KeyError` inside the child | Way 1 with a key the parent never set | `total=False` + `.get()`, or Way 2 with explicit mapping |
| No child events in the stream | `subgraphs=True` missing | Add it; for Way 2 also pass `config` |
| `get_state(...).tasks` is empty | Thread isn't paused inside the child | Only a paused/running child has a snapshot there |
| Interrupt in a child without any checkpointer | Neither graph has one | Give the **parent** a checkpointer |
| Child "forgets" between calls in one thread | Default: per-invocation | `compile(checkpointer=True)` on the child |

## ⚠️ Production notes

> ⚠️ **Production note: subgraphs are your module boundaries.**
> A team owns a subgraph with its own schema and tests (like `mastery.graphs.research`'s research subgraph); the parent only depends on its input and output. Keep the boundary small: it's also what you'll trace and version.

> ⚠️ **Production note: changing a subgraph changes in-flight threads.**
> A thread paused inside a child resumes with the **new** code. Renaming nodes or keys can break those threads: read the backward-compatibility page (this module's research checkpoint) before deploying graph changes.

## 🏋️ Exercises

1. **(Easy)** In Part 1, add a second key `summary` to the child only (a `TypedDict` that extends `Doc`). Does it appear in the parent's output?
2. **(Medium)** Fix the trap in Part 1 without Way 2: give the child its own key `child_log` and copy it into the parent's `log` in a parent node.
3. **(Medium)** In Part 3, resume with `"yes"` instead of `"no"` on a new thread. What's the final text?
4. **(Stretch)** Turn `research_all` (Part 2) into a parallel fan-out with `Send` (Module 07), one `research` call per topic.

<details>
<summary><b>Show solutions</b></summary>

**1.** No: Way 1 passes back only keys that exist in the **parent's** schema. Private child keys stay inside.

**2.**
```python
class ChildState(TypedDict, total=False):
    child_log: Annotated[list[str], operator.add]
child = StateGraph(ChildState).add_node("child_step", lambda s: {"child_log": ["child"]}).add_edge(START, "child_step").compile()

class P(TypedDict, total=False):
    log: Annotated[list[str], operator.add]
    child_log: list[str]
p = (StateGraph(P).add_node("first", lambda s: {"log": ["parent"]}).add_node("child", child)
     .add_node("merge", lambda s: {"log": s["child_log"]})
     .add_edge(START, "first").add_edge("first", "child").add_edge("child", "merge").compile())
print(p.invoke({"log": []})["log"])     # ['parent', 'child']
```

**3.** `Please receive the new firmware.`: `human_check` returns `{}` for "yes".

**4.**
```python
from langgraph.types import Send
class Item(TypedDict):
    topic: str
def one(state: Item):
    return {"notes": [research.invoke({"query": state["topic"]})["summary"]]}
g = (StateGraph(Main).add_node("one", one)
     .add_conditional_edges(START, lambda s: [Send("one", {"topic": t}) for t in s["topics"]], ["one"]).compile())
```
</details>

## ✅ Checkpoint

- [ ] I added a subgraph both ways and can say when to use each.
- [ ] I reproduced and fixed the shared-reducer duplication.
- [ ] I streamed child events with `subgraphs=True` and read a child's paused state.
- [ ] I paused inside a subgraph and resumed from the parent with one checkpointer.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

Read [Use subgraphs](https://docs.langchain.com/oss/python/langgraph/use-subgraphs) and note in [`docs/research-log.md`](../../docs/research-log.md): the three subgraph persistence options the docs describe (per-invocation, per-thread with `checkpointer=True`, stateless with `checkpointer=False`) and when each fits a multi-agent system (Module 09).

## Summary

- Way 1: `add_node("name", compiled_subgraph)` shares keys. Way 2: call it in a node with its own schema.
- Shared append reducers duplicate items; return only new items across the boundary.
- `stream(..., subgraphs=True)` labels events with `ns`; `get_state(cfg, subgraphs=True).tasks[i].state` shows the child.
- One parent checkpointer persists children; interrupts in a child resume from the parent.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 08.06 · Streaming in depth** → [`08-06-streaming.ipynb`](08-06-streaming.ipynb).

[Module 08 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)